# Model Comparison and Hyperparameter Tuning

In [19]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor,
                              ExtraTreesRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, loguniform

In [20]:
# Setup paths
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
clean_meta_path = os.path.join(repo_root, "data", "clean", "metadata_clean.parquet")
raw_wave_path = os.path.join(repo_root, "data", "raw", "waveforms.hdf5")

df = pd.read_parquet(clean_meta_path)

# sample_df = df.sample(n=min(1000, len(df)), random_state=42)

trace_matrix = []
matched = []
with h5py.File(raw_wave_path, "r") as hf:
    data_grp = hf["data"]
    for _, row in df.iterrows():
        b_val, t_idx = str(row["bucket"]).strip(), int(row["tindex"])
        b_key = b_val if b_val.startswith("bucket") else f"bucket{b_val}"
        if b_key in data_grp and t_idx < data_grp[b_key].shape[0]:
            tr = np.array(data_grp[b_key][t_idx, :, :300], dtype = np.float32)
            trace_matrix.append(tr) 
            matched.append(row)

X = np.array(trace_matrix)
matched_df = pd.DataFrame(matched).reset_index(drop=True)
y = matched_df["preferred_source_magnitude"].to_numpy(dtype = np.float32)
# X_waveforms: (number of recordings, 300, 3)
# y: corresponding continuous earthquake magnitudes

# Flatten waveforms for traditional ML models
X_ml = X.reshape(X.shape[0], -1)

print("Waveform shape:", X.shape)
print("ML feature shape:", X_ml.shape)
print("Target shape:", y.shape)

# Verify alignment
assert len(X) == len(y)
assert len(X_ml) == len(y)

# Check missing targets
print("Missing magnitudes:", np.isnan(y).sum())

Waveform shape: (18640, 3, 300)
ML feature shape: (18640, 900)
Target shape: (18640,)
Missing magnitudes: 0


In [21]:
# Proof 20% of the data has magnitude >= 2
print(len(df[df["preferred_source_magnitude"] >= 2]) / len(df))

0.19415236051502147


In [22]:
# Event ID for each waveform
groups = matched_df["event_id"].to_numpy()

# Identify earthquakes by magnitude
test_mask = y >= 2.0
train_mask = y < 2.0

# Identify event IDs in each group
test_events = np.unique(groups[test_mask])
train_events = np.unique(groups[train_mask])

# Check that no event appears in both sets
overlap = np.intersect1d(train_events, test_events)

print("Overlapping events:", len(overlap))

# Select recordings
train_idx = np.where(train_mask)[0]
test_idx = np.where(test_mask)[0]

X_train, X_test = X_ml[train_idx], X_ml[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

groups_train = groups[train_idx]

# Group-aware cross-validation on training data
cv = GroupKFold(n_splits = 5)

print("Training recordings:", len(X_train))
print("Testing recordings:", len(X_test))
print("Training events:", len(train_events))
print("Testing events:", len(test_events))
print("Minimum test magnitude:", y_test.min())


Overlapping events: 0
Training recordings: 15021
Testing recordings: 3619
Training events: 6785
Testing events: 1270
Minimum test magnitude: 2.0


In [23]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 1.3244343996047974
Baseline RMSE: 1.3876374116505212
Baseline R²: -10.23343563079834


In [24]:
models = {"Mean baseline": DummyRegressor(strategy = "mean"),
          "Ridge Regression": Pipeline([("scaler", StandardScaler()),
                                        ("model", Ridge(alpha = 1.0))]),
          "Random Forest": RandomForestRegressor(n_estimators = 200,
                                                 max_features = "sqrt",  # default tests all 900 features per split (very slow)
                                                 random_state = 42,
                                                 n_jobs = -1),
          "Extra Trees": ExtraTreesRegressor(n_estimators = 200,
                                             max_features = "sqrt",
                                             random_state = 42,
                                             n_jobs = -1),
          "HistGradientBoosting": HistGradientBoostingRegressor(random_state = 42),
          "Neural Network (MLP)": Pipeline([("scaler", StandardScaler()),
                                            ("model", 
                                             MLPRegressor(hidden_layer_sizes = (128, 64),
                                                          max_iter = 300,
                                                          early_stopping = True,
                                                          random_state = 42))
                                            ])
}

results = []
fitted_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    predictions = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
        "Training time (seconds)": elapsed
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("RMSE")

print(results_df.to_string(index = False))

/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.221821724494703e-09.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


               Model      MAE     RMSE         R2  Training time (seconds)
HistGradientBoosting 1.277718 1.340318  -9.480360                 2.008475
         Extra Trees 1.292918 1.355802  -9.723913                 3.282561
       Random Forest 1.294356 1.356666  -9.737582                 2.449729
       Mean baseline 1.324434 1.387637 -10.233436                 0.000253
Neural Network (MLP) 1.370719 1.453318 -11.322013                 5.102516
    Ridge Regression 1.379544 1.682814 -15.520868                 0.117113


In [25]:
# Group-aware folds: all recordings of one earthquake stay in the same fold,
# so no event is in both the fitting and validation parts of a fold.
cv = GroupKFold(n_splits = 3)

# Candidate models and hyperparameter distributions
search_spaces = {"Ridge Regression": (Pipeline([("scaler", StandardScaler()),
                                                ("model", Ridge())]),
                                                {"model__alpha": loguniform(1e-3, 1e3)}),
                 "Random Forest": (RandomForestRegressor(random_state = 42,
                                                         n_jobs = 1),
                                                         {"n_estimators": randint(100, 201),
                                                          "max_depth": [None, 10, 20, 30],
                                                          "min_samples_split": randint(2, 16),
                                                          "min_samples_leaf": randint(1, 9),
                                                          "max_features": ["sqrt", "log2", 0.1]}),
                 "Extra Trees": (ExtraTreesRegressor(random_state = 42,
                                                     n_jobs = 1),
                                                     {"n_estimators": randint(100, 201),
                                                      "max_depth": [None, 10, 20, 30],
                                                      "min_samples_split": randint(2, 16),
                                                      "min_samples_leaf": randint(1, 9),
                                                      "max_features": ["sqrt", "log2", 0.1]}),
                 "HistGradientBoosting": (HistGradientBoostingRegressor(random_state = 42),
                                          {"learning_rate": loguniform(0.01, 0.2),
                                           "max_iter": randint(100, 401),
                                           "max_leaf_nodes": randint(10, 41),
                                           "l2_regularization": loguniform(1e-4, 10)}),
                 "Neural Network (MLP)": (Pipeline([("scaler", StandardScaler()),
                                                    ("model", MLPRegressor(max_iter = 300,
                                                                           early_stopping = True,
                                                                           random_state = 42))]),
                                                   {"model__hidden_layer_sizes": [(64,), 
                                                                                  (128, 64), 
                                                                                  (128, 64, 32), 
                                                                                  (256, 128)],
                                                    "model__alpha": loguniform(1e-5, 1e-1),
                                                    "model__learning_rate_init": loguniform(1e-4, 1e-2),
                                                    "model__batch_size": [32, 64, 128]})}

tuned_models = {}
tuning_results = []

for name, (model, params) in search_spaces.items():
    print(f"\nTuning {name}...")
    search = RandomizedSearchCV(estimator = model,
                                param_distributions = params,
                                n_iter = 12,
                                scoring = "neg_root_mean_squared_error",
                                cv = cv,
                                random_state = 42,
                                n_jobs = 6,  # physical cores; 12 workers used too much memory
                                verbose = 1)

    start = time.time()
    search.fit(X_train, y_train, groups = groups_train)
    elapsed = time.time() - start

    tuned_models[name] = search.best_estimator_

    tuning_results.append({"Model": name,
                           "Best CV RMSE": -search.best_score_,
                           "Tuning time (seconds)": elapsed,
                           "Best parameters": search.best_params_})

    print("Best CV RMSE:", -search.best_score_)
    print("Best parameters:", search.best_params_)

tuning_df = pd.DataFrame(tuning_results)
print(tuning_df.to_string(index = False))


Tuning Ridge Regression...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.708196392286482e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.38312112452877e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Best CV RMSE: 0.5284969210624695
Best parameters: {'model__alpha': np.float64(659.871107205407)}

Tuning Random Forest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best CV RMSE: 0.5184130033605344
Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 148}

Tuning Extra Trees...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best CV RMSE: 0.5177617779099505
Best parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 7, 'min_samples_split': 12, 'n_estimators': 171}

Tuning HistGradientBoosting...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/Users/meredithcarter/ESS_469/MLGEO_AU26_seismology/mlgeo-book/.pixi/envs/default/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best CV RMSE: 0.5181911982558338
Best parameters: {'l2_regularization': np.float64(0.06553013900933982), 'learning_rate': np.float64(0.03172768832582949), 'max_iter': 152, 'max_leaf_nodes': 11}

Tuning Neural Network (MLP)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best CV RMSE: 0.5187320311864217
Best parameters: {'model__alpha': np.float64(0.05678201970293124), 'model__batch_size': 64, 'model__hidden_layer_sizes': (256, 128), 'model__learning_rate_init': np.float64(0.0096476850757201)}
               Model  Best CV RMSE  Tuning time (seconds)                                                                                                                                          Best parameters
    Ridge Regression      0.528497               7.227270                                                                                                                       {'model__alpha': 659.871107205407}
       Random Forest      0.518413             132.674375       

In [26]:
final_results = []

for name, model in tuned_models.items():

    predictions = model.predict(X_test)

    final_results.append({"Model": name,
                          "Test MAE": mean_absolute_error(y_test, predictions),
                          "Test RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
                          "Test R2": r2_score(y_test, predictions)})

final_df = pd.DataFrame(final_results)

baseline_predictions = fitted_models["Mean baseline"].predict(X_test)
baseline_result = pd.DataFrame([{"Model": "Mean baseline",
                                 "Test MAE": mean_absolute_error(y_test, baseline_predictions),
                                 "Test RMSE": np.sqrt(mean_squared_error(y_test, baseline_predictions)),
                                 "Test R2": r2_score(y_test, baseline_predictions)}])

final_df = pd.concat([baseline_result, final_df],
                     ignore_index = True).sort_values("Test RMSE")

print(final_df.to_string(index = False))

               Model  Test MAE  Test RMSE    Test R2
         Extra Trees  1.275932   1.337744  -9.440142
       Random Forest  1.278725   1.340314  -9.480303
HistGradientBoosting  1.284320   1.345486  -9.561338
Neural Network (MLP)  1.297339   1.364453  -9.861193
       Mean baseline  1.324434   1.387637 -10.233436
    Ridge Regression  1.330381   1.406925 -10.547878


We decided to use and tune a Random Forest model, an Extra Trees model, a Ridge Regression, a Gradient Boosting Decision Tree, and a Neural Network, then compared them to our baseline of the mean. Before tuning, they all had a similar MAE and RMSE (excluding the Neural Network and Ridge Regression - which were both larger and the Ridge Regression significantly so). Additionally, the $R^2$ value of most of the models was similar, except for the mean baseline, Neural Network, and Ridge Regression which were negative (the Ridge Regression had a value of -14).

Once tuned, the RMSE shrank for all models, and only the Ridge Regression had a comparitively large value. The best value of $\alpha$ (the hyperparameter controlling the strength of the penalty) was found to be 659.87, which resulted in an RMSE of 0.53 after cross validation.

After this point, all tuned models resulted in an RMSE value of 0.52 after cross validation.

After tuning, we found that the optimal $\alpha$ (or penalty weight) is 0.06, the optimal batch size is 64 (or the number of training data values the model cycles through before updating its weights), the size of the hidden layers was 256 for the first layer and 128 for the second, and the best initial learning rate was 0.01.

The Random Forest model was optimized to have a maximum of 10 levels within a decision tree, then chooses a subset of size $\sqrt{\text{Number of Features}}$, a minimum number of samples in each leaf (final subsection) being greater than 4, a minimum number of samples required to split data into branches being greater than 10, and a total number of decision trees created of 148.

Our Extra Trees model (which is similar to a Random Forest model, but without boostrapping) was optimized to have a maximum of 20 levels within each decision tree, then chooses a subset of size $\sqrt{\text{Number of Features}}$, at least 7 samples in each leaf, more than 12 samples required to make a new branch, and a total number of decision trees created of 148.

Finally, our Histogram-Based Gradient Boosting Tree (simiarly to a Random Forest, but based on histograms) was optimized to have a pentalty for leaf weights (to reduce overfitting) of 0.07, a learning rate of 0.03, at most 152 learning trees, and a maximum number of leaf nodes (final results) of 11.

When used on the test data (earthquakes over magnitude 2), the Histogram-Based Gradient Boosting Tree, Random Forest, Extra Trees, and Neural Network all had a similar MAE (between 1.275 and 1.3), a similar RMSE (1.33 to 1.36), and an $R^2$ between -9.83 and -9.44. Additionally, the mean baseline and Ridge Regression achieved similar, albeit less accurate values. Due to this, we will continue our research using the 4 best models - the Extra Trees Model, the Random Forest Model, the Histogram-Based Gradient Boosting Tree, and the Neural Network. However, if interpretability is the most important factor, the mean baseline is also almost similar in accuracy and would also be a good option.